# TP Méthodes d'Ensemble - Partie 4 : Stacking

Architecture de Stacking à deux niveaux : modèles de base (Niveau 0) → méta-modèle (Niveau 1).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
import xgboost as xgb

## 0. Chargement des données

Prérequis : voir `README.md`. On utilise les données **scalées** pour Logistic Regression et XGBoost.

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')
df = df.dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1).values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset chargé : {X_train.shape[0]} train, {X_test.shape[0]} test")

Dataset chargé : 237 train, 60 test


## 1. Stacking avec Sklearn

- **Niveau 0** : Logistic Regression, Random Forest, XGBoost  
- **Niveau 1** : Logistic Regression (méta-modèle)

In [3]:
estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)),
    ('xgb', xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss', random_state=42))
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42)
)

stacking.fit(X_train_scaled, y_train)
y_pred_stacking = stacking.predict(X_test_scaled)
acc_stacking = accuracy_score(y_test, y_pred_stacking)
print(f"StackingClassifier : accuracy = {acc_stacking:.4f}")

StackingClassifier : accuracy = 0.8333


### Comparaison avec les modèles individuels

In [4]:
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

model_rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model_rf.fit(X_train_scaled, y_train)

model_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss', random_state=42)
model_xgb.fit(X_train_scaled, y_train)

print("Modèles individuels (sur données scalées) :")
print(f"   Logistic Regression : {accuracy_score(y_test, model_lr.predict(X_test_scaled)):.4f}")
print(f"   Random Forest        : {accuracy_score(y_test, model_rf.predict(X_test_scaled)):.4f}")
print(f"   XGBoost              : {accuracy_score(y_test, model_xgb.predict(X_test_scaled)):.4f}")
print(f"\nStacking (LR + RF + XGB → LR) : {acc_stacking:.4f}")

Modèles individuels (sur données scalées) :
   Logistic Regression : 0.8333
   Random Forest        : 0.8167
   XGBoost              : 0.8167

Stacking (LR + RF + XGB → LR) : 0.8333
